# Parameter Sensitivity

Dedicated one-parameter sensitivity workflow. The default values reproduce the configuration formerly embedded in `run_model.ipynb`.

In [7]:
# Setup
%load_ext autoreload
%autoreload 2

from config import MACRO_COLUMNS, SCENARIO_PRESETS
from src.notebook_workflow import NotebookRunConfig, build_country_config, prepare_data
from src.sensitivity import run_parameter_sensitivity
from src.visual_helpers import plot_sensitivity, plot_sensitivity_summary

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Inputs

In [ ]:
RUN_SENSITIVITY = True
SCENARIO_NAME = "calibrated_consumption"
run_config = NotebookRunConfig(seed=332, t_max=50, country_iso3="FRA", force_rebuild_data=True)
parameter_path = "central_bank.taylor_rule_overrides['rho']"
# parameter_path = "firms.functions.wage_setter.parameters['labour_market_tightness_markup_scale']"

parameter_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
seeds = [
    18,
    21,
    26,
    32,
    377,
    443,
    435,
    427,
    144,
]
N_JOBS = 10
BATCH_SIZE = 1

## Prepare shared inputs

In [9]:
prepared = prepare_data(run_config)
COUNTRY = prepared.cfg.country_iso3
country_configurations = build_country_config(
    data=prepared.data,
    config=run_config,
    overrides=SCENARIO_PRESETS[SCENARIO_NAME],
)

{'seed': 332, 'country': 'FRA', 't_max': 50, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'DefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
                                  'hiring_speed': 0

## Run

In [10]:
# sensitivity = True
if RUN_SENSITIVITY:
    sensitivity = run_parameter_sensitivity(
        datawrapper=prepared.data,
        country_configurations=country_configurations,
        country_code=COUNTRY,
        parameter_path=parameter_path,
        parameter_values=parameter_values,
        seeds=seeds,
        t_max=prepared.cfg.t_max,
        n_jobs=N_JOBS,
        backend="loky",
        batch_size=BATCH_SIZE,
    )
else:
    print("Set RUN_SENSITIVITY = True to run the experiment.")

## Plots

In [11]:
MACRO_SUMMARY_COLUMNS = ["real_gdp", "cpi_transaction_yoy_change", "unemployment_rate", "central_bank_policy_rate"]

plot_sensitivity_summary(sensitivity, cols=MACRO_SUMMARY_COLUMNS)
plot_sensitivity(sensitivity, cols=MACRO_SUMMARY_COLUMNS)

In [12]:
if sensitivity is not None:
    plot_sensitivity_summary(sensitivity, cols=list(MACRO_COLUMNS))
    plot_sensitivity(sensitivity, cols=list(MACRO_COLUMNS))